# Notebook for practical exercises | Week #4 Lesson #1

## Introduction

This notebook contains practical hands-on exercises for the lesson about *Convolutional Neural Networks for Classification and Detection*.

After this session, you will know how to fine‑tune a pretrained CNN on a real-world clinical problem:
* Classifying pneumonia vs. healthy control from chest X-rays using different model architectures.
* Building a model to detect pneumonia from chest X-rays using different model architectures.
* Reporting evaluation metrics and justifying the choice of model architecture.

## Dataset

We will use the dataset from the [RSNA Pneumonia Detection Challenge](https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge/overview).

In this challenge competitors are predicting whether pneumonia exists in a given image. They do so by predicting bounding boxes around areas of the lung. Samples without bounding boxes are negative and contain no definitive evidence of pneumonia. Samples with bounding boxes indicate evidence of pneumonia.

When making predictions, competitors should predict as many bounding boxes as they feel are necessary, in the format:
confidence x-min y-min width height

All provided images are in DICOM format. Data fields availables in the `labels.csv` file are the following:
* patientId _- A patientId. Each patientId corresponds to a unique image.
* x_ - the upper-left x coordinate of the bounding box.
* y_ - the upper-left y coordinate of the bounding box.
* width_ - the width of the bounding box.
* height_ - the height of the bounding box.
* Target_ - the binary Target, indicating whether this sample has evidence of pneumonia.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

!pip install pydicom
import pydicom
import time
import cv2
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import tv_tensors
import torchvision
from torchvision.transforms import v2 as T
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import roc_auc_score

### Download

If needed, run the following cells to download and unzip the dataset.

<div class="alert alert-block alert-danger">
<b>Replace the <code>DATA_PATH</code> with the path where you want to store the data folder. By default, it will be stored at the root of this repository.</b> <br>
<b>If you have already downloaded the dataset, comment the following cell by adding a <code>#</code> before the <code>!</code></b>
</div>

In [2]:
DATA_PATH = '../..'

if not os.path.exists(f'{DATA_PATH}/data'):
    os.mkdir(f'{DATA_PATH}/data')

!curl https://uni-bonn.sciebo.de/s/qAeDDC5HkigwF5c/download --output {DATA_PATH}/data/rsna-pneumonia-detection-challenge.zip
!unzip {DATA_PATH}/data/rsna-pneumonia-detection-challenge.zip -d {DATA_PATH}/data

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  4 3385M    4  163M    0     0  12.5M      0  0:04:30  0:00:13  0:04:17 17.4M^C
Archive:  ../../data/rsna-pneumonia-detection-challenge.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of ../../data/rsna-pneumonia-detection-challenge.zip or
        ../../data/rsna-pneumonia-detection-challenge.zip.zip, and cannot find ../../data/rsna-pneumonia-detection-challenge.zip.ZIP, period.


<div class="alert alert-block alert-danger">
<b>In case you have low computational power, you can run the train cells only to verify they are correctly running, and use the pretrained models for inference. Use the cell below to download them.</b> <br>
</div>

In [ ]:
!curl https://uni-bonn.sciebo.de/s/FM6ZRi7FJoxnqHL/download --output ./alexnet-pneumonia-lr1e-4-b32-e5.pth
!curl https://uni-bonn.sciebo.de/s/mWpjpgKnQybsFkd/download --output ./efficientnetb0-pneumonia-lr1e-4-b32-e5.pth
!curl https://uni-bonn.sciebo.de/s/NmqCq7eqFTPPnwR/download --output ./resnet18-pneumonia-lr1e-4-b32-e5.pth
!curl https://uni-bonn.sciebo.de/s/FQRxfSJcxjnHwra/download --output ./swin-pneumonia-lr1e-4-b32-e5.pth

!curl https://uni-bonn.sciebo.de/s/KXRsjiFrSLGoKAa/download --output ./fasterrcnn_resnet50_fpn.pth
!curl https://uni-bonn.sciebo.de/s/7KoKfxCqSDsPZZ2/download --output ./retinanet_resnet50_fpn.pth

## Classification

This week, to learn how to work with deep models such as CNNs, we will use the PyTorch library.

### Datasets in PyTorch

PyTorch has two primitives to work with data: `torch.utils.data.DataLoader` and `torch.utils.data.Dataset`. Dataset stores the samples and their corresponding labels, and DataLoader wraps an iterable around the Dataset.
PyTorch offers domain-specific libraries such as TorchText, TorchVision, and TorchAudio, all of which include datasets. For this tutorial, we will be using a custom TorchVision dataset.

In [3]:
class ClassificationDataset(Dataset):
    def __init__(self, dataframe, image_dir, transforms=None):
        super().__init__()

        self.image_ids = dataframe['patientId'].unique()
        self.df = dataframe
        self.image_dir = image_dir
        self.transforms = transforms

    def __getitem__(self, index: int):

        image_id = self.image_ids[index]
        records = self.df[self.df['patientId'] == image_id]

        image_dicom = pydicom.dcmread(f'{self.image_dir}/{image_id}.dcm')
        image = image_dicom.pixel_array
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB).astype(np.float32)
        image /= 255.0
        image = np.transpose(image, (2, 0, 1))
        image = tv_tensors.Image(image)
        # Define the transform
        resize_transform = torchvision.transforms.Resize((64, 64))
        # Apply the transform
        resized_img = resize_transform(image)

        # there is only one class
        labels = torch.tensor(records['Target'].values[0])

        return image, labels

    def __len__(self) -> int:
        return self.image_ids.shape[0]

Transforms are modifications made to the samples during loading. During training, it can include some data augmentation strategies. 

In [4]:
def get_transform(train):
    transforms = []
    if train:
        transforms.append(T.RandomHorizontalFlip(0.5))
    transforms.append(T.ToDtype(torch.float, scale=True))
    transforms.append(T.ToPureTensor())
    return T.Compose(transforms)

In [5]:
image_dir = f'{DATA_PATH}/data/rsna-pneumonia-detection-challenge/images'
dataset_df = pd.read_csv(f'{DATA_PATH}/data/rsna-pneumonia-detection-challenge/labels.csv')
dataset_df = dataset_df.sample(2000)
patient_list = dataset_df['patientId'].unique()

patient_train, patient_test = train_test_split(
    patient_list, # List or array to split
    test_size=0.2, # Size of the subset
    random_state=42)

patient_train, patient_val = train_test_split(
    patient_train,
    test_size=0.1 / 0.8,
    random_state=42)

train_df = dataset_df.loc[dataset_df['patientId'].isin(patient_train)]
val_df = dataset_df.loc[dataset_df['patientId'].isin(patient_val)]

In [6]:
train_dataset = ClassificationDataset(train_df, image_dir, transforms=get_transform(True))

<div class="alert alert-block alert-info">
<b>Q1.</b> Similarly, create a <code>val_dataset</code> variable to store the validation set.
</div>

In [7]:
val_dataset = ... # COMPLETE

We pass the `Dataset` as an argument to `DataLoader`. This wraps an iterable over our dataset, and supports automatic batching, sampling, shuffling and multiprocess data loading. Here we define a batch size of 32, *i.e.* each element in the dataloader iterable will return a batch of 32 features and labels.

In [8]:
batch_size = 32

# Create data loaders.
train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

for X, y in train_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([32, 3, 1024, 1024])
Shape of y: torch.Size([32]) torch.int64


### Loading pretrained models with PyTorch

The `torchvision.models` subpackage contains definitions of models for addressing different tasks, including: image classification, pixelwise semantic segmentation, object detection, instance segmentation, person keypoint detection, video classification, and optical flow.

TorchVision offers a new mechanism which allows listing and retrieving models and weights by their names. Here are a few examples on how to use them:

In [9]:
torchvision.models.list_models()

['alexnet',
 'convnext_base',
 'convnext_large',
 'convnext_small',
 'convnext_tiny',
 'deeplabv3_mobilenet_v3_large',
 'deeplabv3_resnet101',
 'deeplabv3_resnet50',
 'densenet121',
 'densenet161',
 'densenet169',
 'densenet201',
 'efficientnet_b0',
 'efficientnet_b1',
 'efficientnet_b2',
 'efficientnet_b3',
 'efficientnet_b4',
 'efficientnet_b5',
 'efficientnet_b6',
 'efficientnet_b7',
 'efficientnet_v2_l',
 'efficientnet_v2_m',
 'efficientnet_v2_s',
 'fasterrcnn_mobilenet_v3_large_320_fpn',
 'fasterrcnn_mobilenet_v3_large_fpn',
 'fasterrcnn_resnet50_fpn',
 'fasterrcnn_resnet50_fpn_v2',
 'fcn_resnet101',
 'fcn_resnet50',
 'fcos_resnet50_fpn',
 'googlenet',
 'inception_v3',
 'keypointrcnn_resnet50_fpn',
 'lraspp_mobilenet_v3_large',
 'maskrcnn_resnet50_fpn',
 'maskrcnn_resnet50_fpn_v2',
 'maxvit_t',
 'mc3_18',
 'mnasnet0_5',
 'mnasnet0_75',
 'mnasnet1_0',
 'mnasnet1_3',
 'mobilenet_v2',
 'mobilenet_v3_large',
 'mobilenet_v3_small',
 'mvit_v1_b',
 'mvit_v2_s',
 'quantized_googlenet',
 '

In [10]:
from torchvision.models import resnet18, ResNet18_Weights

# Initialize model with the best available weights
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### Hyperparameters

Hyperparameters are adjustable parameters that let you control the model optimization process. Different hyperparameter values can impact model training and convergence rates.

We define the following hyperparameters for training:
* Number of Epochs - the number of times to iterate over the dataset
* Batch Size - the number of data samples propagated through the network before the parameters are updated
* Learning Rate - how much to update models parameters at each batch/epoch. Smaller values yield slow learning speed, while large values may result in unpredictable behavior during training.

In [11]:
learning_rate = 1e-4
batch_size = 32
epochs = 5

### Optimization loop

Once we set our hyperparameters, we can then train and optimize our model with an optimization loop. Each iteration of the optimization loop is called an epoch.

Each epoch consists of two main parts:
* The Train Loop - iterate over the training dataset and try to converge to optimal parameters.
* The Validation/Test Loop - iterate over the test dataset to check if model performance is improving on unseen data.

When presented with some training data, our untrained network is likely not to give the correct answer. Loss function measures the degree of dissimilarity of obtained result to the target value, and it is the loss function that we want to minimize during training. To calculate the loss we make a prediction using the inputs of our given data sample and compare it against the true data label value.

We pass our model’s output logits to `nn.CrossEntropyLoss`, which will normalize the logits and compute the prediction error.

In [12]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

Optimization is the process of adjusting model parameters to reduce model error in each training step. Optimization algorithms define how this process is performed (in this example we use Stochastic Gradient Descent). All optimization logic is encapsulated in the optimizer object. Here, we use the SGD optimizer; additionally, there are many different optimizers available in PyTorch such as ADAM and RMSProp, that work better for different kinds of models and data.

We initialize the optimizer by registering the model’s parameters that need to be trained, and passing in the learning rate hyperparameter.

In [13]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

Inside the training loop, optimization happens in three steps:
* Call `optimizer.zero_grad()` to reset the gradients of model parameters. Gradients by default add up; to prevent double-counting, we explicitly zero them at each iteration.
* Backpropagate the prediction loss with a call to `loss.backward()`. PyTorch deposits the gradients of the loss w.r.t. each parameter.
* Once we have our gradients, we call `optimizer.step()` to adjust the parameters by the gradients collected in the backward pass.

In [14]:
# train on the GPU or on the CPU, if a GPU is not available
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

def train_loop(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    # Set the model to training mode
    model.train()

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

    return model

def val_loop(dataloader, model, loss_fn, device):
    # Set the model to evaluation mode
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

#### Overall optimization process

<div class="alert alert-block alert-info">
<b>Q2.</b> Run the following cell to visualize the overall optimization process. 
</div>

In [ ]:
learning_rate = 1e-4
batch_size = 32
epochs = 5

weights = ResNet18_Weights.DEFAULT

model_resnet = resnet18(weights=weights)
model_resnet.fc = nn.Linear(512, 2)
preprocess = weights.transforms()

optimizer = torch.optim.Adam(model_resnet.parameters(), lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

train_dataset = ClassificationDataset(train_df, image_dir, transforms=preprocess)
val_dataset = ClassificationDataset(val_df, image_dir,  transforms=preprocess)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

model_resnet.to(device)
loss_fn.to(device)

s = time.process_time() # start time
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    model_resnet = train_loop(train_dataloader, model_resnet, loss_fn, optimizer, device)
    val_loop(val_dataloader, model_resnet, loss_fn, device)
print("Done!")

torch.save(model_resnet.state_dict, './resnet18-pneumonia-lr1e-4-b32-e5.pth')

e = time.process_time() # end time
print(e - s, "seconds")

Epoch 1
-------------------------------
loss: 0.831538  [   32/ 1389]


In [ ]:
#model_resnet.load_state_dict(torch.load('./resnet18-pneumonia-lr1e-4-b32-e5.pth', map_location=device)) #Uncomment to use pretrained models

### Benchmarking model architectures

CNN architectures differ primarily in their layer arrangements, use of skip connections, and strategies for dealing with vanishing gradients, impacting their ability to scale and learn complex features. Some architectures, like ResNet, use skip connections to bypass layers, enabling deeper networks without vanishing gradients, while others, like DenseNet, employ dense connectivity within blocks. Other variations include Inception modules for parallel processing and feature combination, and normalization layers to stabilize training.

In the following, you will train different model architecture and compare the performance you obtain on the validation set.

<div class="alert alert-block alert-info">
<b>Q3.</b> Complete the following cells to train other model architectures: <br>
    - AlexNet <br>
    - EfficientNet <br>
    - Swin transformer <br>
</div>

#### AlexNet

In [ ]:
from torchvision.models import alexnet, AlexNet_Weights

learning_rate = ... # COMPLETE
batch_size = ... # COMPLETE
epochs = ... # COMPLETE

model_alexnet = ... 
model_alexnet.classifier[6] = ... # COMPLETE

preprocess = ... # COMPLETE

train_dataset = ... # COMPLETE
val_dataset = ... # COMPLETE

train_dataloader = ... # COMPLETE
val_dataloader = ... # COMPLETE

loss_fn = ... # COMPLETE
optimizer = ... # COMPLETE

model_alexnet.to(device)
loss_fn.to(device)

s = time.process_time() # start time
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    model_alexnet = train_loop(train_dataloader, model_alexnet, loss_fn, optimizer, device)
    val_loop(val_dataloader, model_alexnet, loss_fn, device)
print("Done!")

torch.save(model_alexnet.state_dict, './alexnet-pneumonia-lr1e-4-b32-e5.pth')

e = time.process_time() # end time
print(e - s, "seconds")

In [ ]:
#model_alexnet.load_state_dict(torch.load('./alexnet-pneumonia-lr1e-4-b32-e5.pth', map_location=device)) #Uncomment to use pretrained models

#### EfficientNet

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

learning_rate = ... # COMPLETE
batch_size = ... # COMPLETE
epochs = ... # COMPLETE

model_enb0 = ... # COMPLETE
model_enb0.classifier[1] = ... # COMPLETE

preprocess = ... # COMPLETE

train_dataset = ... # COMPLETE
val_dataset = ... # COMPLETE

train_dataloader = ... # COMPLETE
val_dataloader = ... # COMPLETE

loss_fn = ... # COMPLETE
optimizer = ... # COMPLETE

model_enb0.to(device)
loss_fn.to(device)

s = time.process_time() # start time
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    model_enb0 = train_loop(train_dataloader, model_enb0, loss_fn, optimizer, device)
    val_loop(val_dataloader, model_enb0, loss_fn, device)
print("Done!")

torch.save(model_enb0.state_dict, './efficientnetb0-pneumonia-lr1e-4-b32-e5.pth')

e = time.process_time() # end time
print(e - s, "seconds")

In [ ]:
#model_enb0.load_state_dict(torch.load('./efficientnetb0-pneumonia-lr1e-4-b32-e5.pth', map_location=device)) #Uncomment to use pretrained models

#### Swin transformer

In [ ]:
from torchvision.models import swin_t, Swin_T_Weights

learning_rate = ... # COMPLETE
batch_size = ... # COMPLETE
epochs = ... # COMPLETE

model_swin = ... # COMPLETE
model_swin.head = ... # COMPLETE

preprocess = ... # COMPLETE

train_dataset = ... # COMPLETE
val_dataset = ... # COMPLETE

train_dataloader = ... # COMPLETE
val_dataloader = ... # COMPLETE

loss_fn = ... # COMPLETE
optimizer = ... # COMPLETE

model_swin.to(device)
loss_fn.to(device)

s = time.process_time() # start time
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    model_swin = train_loop(train_dataloader, model_swin, loss_fn, optimizer, device)
    val_loop(val_dataloader, model_swin, loss_fn, device)
print("Done!")

torch.save(model_swin.state_dict, './swint-pneumonia-lr1e-4-b32-e5.pth')

e = time.process_time() # end time
print(e - s, "seconds")

In [ ]:
#model_swin.load_state_dict(torch.load('./swint-pneumonia-lr1e-4-b32-e5.pth', map_location=device)) # Uncomment to use pretrained models

#### Metrics

Now that we have trained our models, we can evaluate their performance on the validation set using different metrics to select the best performing one.

<div class="alert alert-block alert-info">
<b>Q4.</b> Compare the performance of the four models using appropriate metrics: <br>
    - Plot the ROC curve <br>
    - Accuracy score <br> 
    - Precision and Recall <br> 
    - F1-score <br> 
    - etc.
</div>

In [ ]:
for model in [model_resnet, model_alexnet, model_enb0, model_swin]:
    model.eval()
    model.to(device)
    with torch.no_grad():
    for X, y in val_dataloader:
        X, y = X.to(device), y.to(device)
        out = model(X).detach().cpu()
        probs = nn.functional.softmax(out, dim=1)
        preds = probs.max(1)[1].numpy()
        labels = y.detach().cpu().numpy()
    
    ... # COMPLETE WITH ROC Curve display, and other metrics

<div class="alert alert-block alert-info">
<b>Conclude:</b> Which model gives the best performance and should be selected?
</div>

## Detection

Now that we have seen how to perform classification, we will train a detection model. 
In classification, our goal was simply to assign a single label to the entire input (e.g., “normal” vs. “pneumonia”), relying on feature
extractors like ResNet or EfficientNet to encode global image cues. By contrast, detection requires both identifying and localizing multiple regions of interest—such as nodules, organs, or lesions—within a single image.

In [ ]:
from torchvision.utils import draw_bounding_boxes
os.system("wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/engine.py")
os.system("wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/utils.py")
os.system("wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/coco_utils.py")
os.system("wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/coco_eval.py")
os.system("wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/transforms.py")
import utils
from engine import train_one_epoch, evaluate

For detection, `Dataset` classes must be constructed differently. Observe the following one:

In [ ]:
class PredictionDataset(Dataset):
    def __init__(self, dataframe, image_dir, transforms=None):
        super().__init__()

        self.image_ids = dataframe['patientId'].unique()
        self.df = dataframe
        self.image_dir = image_dir
        self.transforms = transforms

    def __getitem__(self, index: int):

        image_id = self.image_ids[index]
        records = self.df[self.df['patientId'] == image_id]

        image_dicom = pydicom.dcmread(f'{self.image_dir}/{image_id}.dcm')
        image = image_dicom.pixel_array
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB).astype(np.float32)
        image /= 255.0
        image = np.transpose(image, (2, 0, 1))
        image = tv_tensors.Image(image)
        # Define the transform
        resize_transform = torchvision.transforms.Resize((256, 256))
        # Apply the transform
        resized_img = resize_transform(image)

        boxes = records[['x', 'y', 'width', 'height']].values
        boxes[:,2] = boxes[:,0] + boxes[:,2]
        boxes[:,3] = boxes[:,1] + boxes[:,3]

        area = (boxes[:,3] - boxes[:,1]) * (boxes[:,2] - boxes[:,0])
        area = torch.as_tensor(area, dtype=torch.float64)

        # there is only one class
        labels = torch.ones((records.shape[0],), dtype=torch.int64)

        # suppose all instances are not crowd
        iscrowd = torch.zeros((records.shape[0],), dtype=torch.int64)

        target = {}
        target['boxes'] = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(1024,1024))
        target['labels'] = labels
        target['patientId'] = torch.tensor([index])
        target['area'] = area
        target['iscrowd'] = iscrowd

        if self.transforms:
            image, target = self.transforms(image, target)

        return image, target

    def __len__(self) -> int:
        return self.image_ids.shape[0]

In our dataset, we have both healthy and pneumonia samples. For classification, we needed to use both. But for detection, we need to filter the dataset to only keep pneumonia samples, as the healthy ones don't have any bounding box labels.

<div class="alert alert-block alert-info">
<b>Q5.</b> Explore the structure of the labels with bounding boxes. How many positive samples (with pneumonia) are in the dataset? What proportion of images contain multiple bounding boxes?
</div>

In [ ]:
# N. of positive samples
positive_samples = ... # COMPLETE
total_samples = ... # COMPLETE
print(f"Positive cases: {positive_samples} / {total_samples}")

In [ ]:
# Images with multiple bbox
multiple_boxes = ... # COMPLETE
print(f"Images with >1 bounding box: {(multiple_boxes > 1).sum()}")

<div class="alert alert-block alert-info">
<b>Q6.</b> Here, we visualize the bounding box on an example image. What challenges might arise in detecting these regions?
</div>

In [ ]:
patient_df = dataset_df[dataset_df.Target == 1].iloc[0]
img_path = os.path.join(image_dir, "{}.dcm".format(patient_df.patientId))

plt.imshow(pydicom.dcmread(img_path).pixel_array)

img_size = 1024
x, y, w, h = patient_df[['x', 'y', 'width', 'height']].values
plt.plot([x, x, x+w, x+w, x], [y, y+h, y+h, y, y], 'red')

In this exercise, we will use Faster R-CNN. This network integrates a Region Proposal Network (RPN) directly into a two-stage detection framework. In the first stage, the RPN slides small convolutional windows over the shared feature map to generate objectness scores and bounding-box proposals. In the second stage, each proposal is cropped and reshaped via RoI pooling (or RoI Align) and passed through fully connected layers to predict refined box coordinates and classification probabilities. 

In [ ]:
train_df_pos = train_df[train_df.Target == 1]
val_df_pos = val_df[val_df.Target == 1]

train_dataset = PredictionDataset(train_df_pos, image_dir, transforms=get_transform(True))
val_dataset = PredictionDataset(val_df_pos, image_dir,  transforms=get_transform(False))

train_dataloader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn = utils.collate_fn
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn = utils.collate_fn
)

Let's observe what is the output of the model:

In [ ]:
# For Training
images, targets = next(iter(train_dataloader))
images = list(image for image in images)
targets = [{k: v for k, v in t.items()} for t in targets]
output = model(images, targets)  # Returns losses and detections
print(output)

<div class="alert alert-block alert-info">
<b>Q6.</b> The bounding boxes are provided as (<code>x, y, width, height</code>). In the <code>Dataset</code> class, we convert them to (<code>x_min, y_min, x_max, y_max</code>) format. Why might this format be useful in practice?
</div>

Here, we train a FASTER-RCNN model for detection of pneumonia. Observe the following code:

In [ ]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

In [ ]:
# train on the GPU or on the CPU, if a GPU is not available
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# our dataset has two classes only - background and pneumonia
num_classes = 2

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

num_epochs = 5
model.to(device)
for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, train_dataloader, device, epoch, print_freq=1000)
    # update the learning rate
    lr_scheduler.step()

torch.save(model.state_dict(), './fasterrcnn_resnet50_fpn.pth')

<div class="alert alert-block alert-info">
<b>Q7.</b> Similarly, complete the following code to train a RetinaNet model. 
</div>

In [ ]:
from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

# Load pretrained RetinaNet
model = ... # COMPLETE

# Modify the classification head for 2 classes (background + pneumonia)
num_classes = 2
in_features = model.head.classification_head.conv[0][0].in_channels
num_anchors = model.head.classification_head.num_anchors

model.head.classification_head = RetinaNetClassificationHead(
    in_channels=in_features,
    num_anchors=num_anchors,
    num_classes=num_classes
)

# How many classes do we have?
num_classes = ... # COMPLETE

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = ... # COMPLETE

# and a learning rate scheduler
lr_scheduler = ... # COMPLETE

num_epochs = 5
model.to(device)
for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, train_dataloader, device, epoch, print_freq=1000)
    # update the learning rate
    lr_scheduler.step()

torch.save(model.state_dict(), './retinanet_resnet50_fpn.pth')

In [ ]:
model.load_state_dict(torch.load('./retinanet_resnet50_fpn.pth', map_location='cpu'))

In [ ]:
def calculate_precision(gts, preds, threshold = 0.5, form = 'coco', ious=None) -> float:
    # https://www.kaggle.com/sadmanaraf/wheat-detection-using-faster-rcnn-train
    """Calculates precision for GT - prediction pairs at one threshold.

    Args:
        gts: (List[List[Union[int, float]]]) Coordinates of the available ground-truth boxes
        preds: (List[List[Union[int, float]]]) Coordinates of the predicted boxes,
               sorted by confidence value (descending)
        threshold: (float) Threshold
        form: (str) Format of the coordinates
        ious: (np.ndarray) len(gts) x len(preds) matrix for storing calculated ious.

    Return:
        (float) Precision
    """
    n = len(preds)
    tp = 0
    fp = 0

    for pred_idx in range(n):

        best_match_gt_idx = find_best_match(gts, preds[pred_idx], pred_idx,
                                            threshold=threshold, form=form, ious=ious)

        if best_match_gt_idx >= 0:
            # True positive: The predicted box matches a gt box with an IoU above the threshold.
            tp += 1
            # Remove the matched GT box
            gts[best_match_gt_idx] = -1
        else:
            # No match
            # False positive: indicates a predicted box had no associated gt box.
            fp += 1

    # False negative: indicates a gt box had no associated predicted box.
    fn = (gts.sum(axis=1) > 0).sum()

    return tp / (tp + fp + fn)
    
def calculate_image_precision(gts, preds, thresholds = (0.5, ), form = 'coco') -> float:
    # https://www.kaggle.com/sadmanaraf/wheat-detection-using-faster-rcnn-train
    """Calculates image precision.

    Args:
        gts: (List[List[Union[int, float]]]) Coordinates of the available ground-truth boxes
        preds: (List[List[Union[int, float]]]) Coordinates of the predicted boxes,
               sorted by confidence value (descending)
        thresholds: (float) Different thresholds
        form: (str) Format of the coordinates

    Return:
        (float) Precision
    """
    n_threshold = len(thresholds)
    image_precision = 0.0

    ious = np.ones((len(gts), len(preds))) * -1
    # ious = None

    for threshold in thresholds:
        precision_at_threshold = calculate_precision(gts.copy(), preds, threshold=threshold,
                                                     form=form, ious=ious)
        image_precision += precision_at_threshold / n_threshold

    return image_precision

In [ ]:
precisions = []
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
model.load_state_dict(torch.load('./fasterrcnn_resnet50_fpn.pth'))
model.eval()
with torch.no_grad():
  for images, targets in val_dataloader:
    images = list(image.to(device) for image in images)
    targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]
    outputs = model(images)
    outputs = [{k: v.to('cpu') for k, v in t.items()} for t in outputs]
    precisions.append(calculate_image_precision(targets[0]['boxes'].detach().cpu().numpy(), outputs[0]['boxes'].detach().cpu().numpy()))

  print('Average precision:', np.mean(precisions))

<div class="alert alert-block alert-info">
<b>Q8.</b> Compare the precision of the two models. Which one performs best? Why?
</div>

In [ ]:
# Load pretrained RetinaNet
model = ... # COMPLETE
model # COMPLETE to load state dictionary

# Modify the classification head for 2 classes (background + pneumonia)
num_classes = 2
in_features = model.head.classification_head.conv[0][0].in_channels
num_anchors = model.head.classification_head.num_anchors

model.head.classification_head = RetinaNetClassificationHead(
    in_channels=in_features,
    num_anchors=num_anchors,
    num_classes=num_classes
)

precisions = []

model.eval()
with torch.no_grad():
  for images, targets in val_dataloader:
    images = list(image.to(device) for image in images)
    targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]
    outputs = model(images)
    outputs = [{k: v.to('cpu') for k, v in t.items()} for t in outputs]
    precisions.append(calculate_image_precision(targets[0]['boxes'].detach().cpu().numpy(), outputs[0]['boxes'].detach().cpu().numpy()))

  print('Average precision:', np.mean(precisions))

<div class="alert alert-block alert-info">
<b>Q9.</b> The RSNA challenge uses IoU (Intersection over Union) and mAP (mean Average Precision). <br> 
    Implement IoU between two bounding boxes and compute the score on validation data. Why is IoU a better fit than accuracy for detection?
</div>

In [ ]:
def compute_iou(box1, box2):
    xA = ... # COMPLETE
    yA = ... # COMPLETE
    xB = ... # COMPLETE
    yB = ... # COMPLETE

    interArea = max(0, xB - xA) * max(0, yB - yA)
    box1Area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2Area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    iou = ... # COMPLETE
    return iou

<div class="alert alert-block alert-info">
<b>Conclude.</b> Compare the performance of the two trained models and justify your selection based on multiple criterias, including computing time. 
</div>

In [ ]:
def evaluate_model(model, dataloader, device, iou_threshold=0.5, conf_threshold=0.5):
    model.eval()
    all_precisions = []
    all_recalls = []
    all_ious = []

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Evaluating"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            outputs = model(images)

            for pred, target in zip(outputs, targets):
                gt_boxes = target['boxes'].cpu().numpy()
                pred_boxes = pred['boxes'].cpu().numpy()
                pred_scores = pred['scores'].cpu().numpy()

                # Filter by confidence
                pred_boxes = pred_boxes[pred_scores > conf_threshold]

                # Compute IoUs
                matched = set()
                ious = []
                for gt_box in gt_boxes:
                    best_iou = 0
                    for i, pred_box in enumerate(pred_boxes):
                        if i in matched:
                            continue
                        iou = compute_iou(gt_box, pred_box)
                        if iou > best_iou:
                            best_iou = iou
                            best_idx = i
                    if best_iou >= iou_threshold:
                        matched.add(best_idx)
                        ious.append(best_iou)

                TP = len(matched)
                FP = len(pred_boxes) - TP
                FN = len(gt_boxes) - TP

                precision = TP / (TP + FP) if (TP + FP) > 0 else 0
                recall = TP / (TP + FN) if (TP + FN) > 0 else 0

                all_precisions.append(precision)
                all_recalls.append(recall)
                all_ious.extend(ious)

    return {
        "precision": np.mean(all_precisions),
        "recall": np.mean(all_recalls),
        "mean_iou": np.mean(all_ious) if all_ious else 0
    }

In [ ]:
# Example usage (assuming you already have both models and a val_loader)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Evaluate Faster R-CNN
faster_rcnn_results = ... # COMPLETE

# Evaluate RetinaNet
retinanet_results = ... # COMPLETE

# Print comparison # COMPLETE
print("Faster R-CNN Results:")
print(f"Precision: ")
print(f"Recall: ")
print(f"Mean IoU: ")

print("RetinaNet Results:")
print(f"Precision: ")
print(f"Recall: ")
print(f"Mean IoU: ")